# Corrected Thermo-Piezoelectric PINN Notebook

In [1]:

# ============================================================
# PROJECT SETUP
# ============================================================

import sys
import os

ROOT = os.getcwd()

if ROOT not in sys.path:
    sys.path.append(ROOT)

print("Project Root:", ROOT)


Project Root: d:\pinn_piezo_crack


In [2]:

# ============================================================
# IMPORTS
# ============================================================

import torch
import numpy as np
import matplotlib.pyplot as plt

from src import config as cfg

from src.network import MechanicsNet
from src.loss import pinn_loss
from src.trainer import Trainer

from src.sampling import (
    sample_domain_points,
    sample_left_face,
    sample_crack_face,
    sample_top_bottom,
    sample_far_field,
)

from src.problem_definition import ACTIVE_PROBLEM

print("All imports successful.")


All imports successful.


In [3]:

# ============================================================
# CONFIGURATION CHECK
# ============================================================

print("Geometry:")
print("H =", cfg.H)
print("L_TRUNC =", cfg.L_TRUNC)

print("\nCrack:")
print("A_CRACK =", cfg.A_CRACK)
print("B_CRACK =", cfg.B_CRACK)

print("\nTime:")
print("T_MAX =", cfg.T_MAX)

print("\nTraining:")
print("N_INTERIOR =", cfg.N_INTERIOR)
print("N_BOUNDARY =", cfg.N_BOUNDARY)

print("\nMaterial:")
print("MU_11 =", cfg.MU_11)
print("MU_33 =", cfg.MU_33)


Geometry:
H = 4.0
L_TRUNC = 10.0

Crack:
A_CRACK = 1.0
B_CRACK = 3.0

Time:
T_MAX = 2.0

Training:
N_INTERIOR = 8000
N_BOUNDARY = 1500

Material:
MU_11 = 139000000000.0
MU_33 = 113000000000.0


In [4]:

# ============================================================
# NETWORK INITIALIZATION
# ============================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.float64

from src.problem_definition import MultiModeNet

net = MultiModeNet().to(device=device, dtype=dtype)

print(net)


MultiModeNet(
  (nets): ModuleList(
    (0-6): 7 x TriclinicNet(
      (net): Sequential(
        (0): Linear(in_features=3, out_features=64, bias=True)
        (1): Tanh()
        (2): Linear(in_features=64, out_features=64, bias=True)
        (3): Tanh()
        (4): Linear(in_features=64, out_features=64, bias=True)
        (5): Tanh()
        (6): Linear(in_features=64, out_features=4, bias=True)
      )
    )
  )
)


In [5]:

# ============================================================
# FORWARD PASS TEST
# ============================================================

x1 = torch.rand(10,1, device=device, dtype=dtype, requires_grad=True)
x3 = torch.rand(10,1, device=device, dtype=dtype, requires_grad=True)
t  = torch.rand(10,1, device=device, dtype=dtype, requires_grad=True)

# ============================================================
# FORWARD PASS TEST
# ============================================================

x1 = torch.rand(
    10, 1,
    device=device,
    dtype=dtype,
    requires_grad=True
)

x3 = torch.rand(
    10, 1,
    device=device,
    dtype=dtype,
    requires_grad=True
)

t = torch.rand(
    10, 1,
    device=device,
    dtype=dtype,
    requires_grad=True
)

# Test first subnet
u1, u2, u3, phi = net.nets[0](x1, x3, t)

print("u1 shape :", u1.shape)
print("u2 shape :", u2.shape)
print("u3 shape :", u3.shape)
print("phi shape:", phi.shape)


u1 shape : torch.Size([10, 1])
u2 shape : torch.Size([10, 1])
u3 shape : torch.Size([10, 1])
phi shape: torch.Size([10, 1])


In [6]:

# ============================================================
# SAMPLING TEST
# ============================================================

x1_int, x3_int, t_int = sample_domain_points(100)

print("Interior shapes:")
print(x1_int.shape, x3_int.shape, t_int.shape)

x1_bc, x3_bc, t_bc = sample_left_face(100)

print("\nBoundary shapes:")
print(x1_bc.shape, x3_bc.shape, t_bc.shape)


Interior shapes:
torch.Size([100, 1]) torch.Size([100, 1]) torch.Size([100, 1])

Boundary shapes:
torch.Size([100, 1]) torch.Size([100, 1]) torch.Size([100, 1])


In [7]:

# ============================================================
# LOSS FUNCTION TEST
# ============================================================

loss, comps = pinn_loss(
    net,
    device=device,
    dtype=dtype,
)

print("Total Loss =", loss.item())

print("\nComponents:")
for k,v in comps.items():
    print(k, ":", float(v))


Total Loss = 26.08579363110449

Components:
bc_u1_sum : 10.968959977151897
bc_u2_sum : 7.945306522911343
bc_u3_sum : 2.6631880379143964
bc_T31_sum : 1.5716174691363827
bc_T32_sum : 0.647694809426788
bc_phi_sum : 2.2117253102093395
bc_total : 26.008492126750145
ic_total : 0.0
pde : 0.07730150435434488
total : 26.08579363110449


In [8]:

# ============================================================
# TRAINER INITIALIZATION
# ============================================================

trainer = Trainer(
    net,
    device=device,
    dtype=dtype,
)

print("Trainer initialized successfully.")


Trainer initialized successfully.


In [9]:

# ============================================================
# SINGLE OPTIMIZATION STEP
# ============================================================

optimizer = torch.optim.Adam(net.parameters(), lr=1e-3)

optimizer.zero_grad()

loss, comps = pinn_loss(
    net,
    device=device,
    dtype=dtype,
)

loss.backward()

optimizer.step()

print("Single optimization step successful.")
print("Loss =", float(loss))


Single optimization step successful.
Loss = 25.71015104204336


C:\Users\Global\AppData\Local\Temp\ipykernel_20280\1115426901.py:20: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\autograd\generated\python_variable_methods.cpp:839.)
  print("Loss =", float(loss))


In [10]:

# ============================================================
# IMPORTANT NOTES
# ============================================================

# Removed incompatible old-project cells:
#
# - CONFIG["LAYER"]
# - CONFIG["HALFSPACE"]
# - get_all_networks()
# - networks.py imports
# - losses.py imports
# - layered-medium geometry
# - z_layer / z_half sampling
#
# This notebook is now fully compatible with:
#
#   network.py
#   loss.py
#   pde_residuals.py
#   trainer.py
#   boundary_conditions.py
#   config.py
#
# from your thermo-piezoelectric crack PINN project.
